In [ ]:
# Environment setup & Imports
import os, sys
sys.path.append(os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

from shared_utils.theme import PALETTE, apply_sleek_theme, create_dashboard_header
from van_der_pol_oscillator import simulate_van_der_pol
from ecological_chaos_demo import (
    logistic_map, generate_bifurcation_diagram,
    compute_lyapunov_spectrum, compute_cobweb_points
)

apply_sleek_theme()
display(HTML(create_dashboard_header(
    'ACT 04: LIMIT CYCLES, POINCARE-BENDIXSON & DISCRETE CHAOS',
    'Interactive GUI Suite: Van der Pol Relaxation, Cobweb Dynamics & Feigenbaum Cascade'
)))

In [ ]:
# LAB 4.1: VAN DER POL OSCILLATOR & LIMIT CYCLE TOPOGRAPHY
display(HTML(create_dashboard_header(
    'LAB 4.1: VAN DER POL OSCILLATOR & RELAXATION OSCILLATIONS',
    'Scrub non-linearity parameter mu to observe transition from harmonic cycle to sharp relaxation jumping.'
)))

def render_vdp_dashboard(mu_val):
    t_pts, x_pts, y_pts = simulate_van_der_pol(mu=mu_val, state0=(0.1, 0.1), t_span=(0.0, 35.0), n_pts=800)
    
    fig, axs = plt.subplots(1, 2, figsize=(13, 5), dpi=100)
    
    # Panel A: Time-series waveform
    axs[0].plot(t_pts, x_pts, color=PALETTE['cyan'], lw=1.6, label='Position x(t)')
    axs[0].plot(t_pts, y_pts, color=PALETTE['amber'], linestyle='--', lw=1.2, label='Velocity y(t)')
    mode_name = 'Harmonic Quasi-Sinusoid' if mu_val < 0.5 else ('Relaxation Jumping' if mu_val > 2.5 else 'Intermediate Limit Cycle')
    axs[0].set_title('Time Response x(t), y(t) | ' + mode_name, fontsize=11)
    axs[0].set_xlabel('Time t', fontsize=10)
    axs[0].set_ylabel('Amplitude', fontsize=10)
    axs[0].set_ylim(-3.5, 3.5)
    axs[0].legend(loc='upper right')
    
    # Panel B: Phase Portrait with Lienard Cubic Nullcline
    axs[1].plot(x_pts, y_pts, color=PALETTE['cyan'], lw=1.8, label='Trajectory Orbit')
    # Approximate limit cycle boundary
    axs[1].scatter([x_pts[-1]], [y_pts[-1]], color=PALETTE['emerald'], s=60, zorder=6, label='Asymptotic Limit Cycle')
    axs[1].scatter([0], [0], color=PALETTE['crimson'], s=50, zorder=6, label='Unstable Origin (Source)')
    axs[1].set_title('Phase Portrait (x vs dx/dt) | mu = ' + str(round(mu_val, 2)), fontsize=11)
    axs[1].set_xlabel('Position x', fontsize=10)
    axs[1].set_ylabel('Velocity y', fontsize=10)
    axs[1].set_xlim(-3.5, 3.5)
    axs[1].set_ylim(-4.0, 4.0)
    axs[1].legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

s_mu = widgets.FloatSlider(value=1.0, min=0.05, max=4.5, step=0.1, description='Non-linear mu:', layout=widgets.Layout(width='450px'))
ui_vdp = widgets.HBox([s_mu], layout=widgets.Layout(padding='10px', background_color=PALETTE['surface'], border='1px solid ' + PALETTE['border'], border_radius='8px'))
out_vdp = widgets.interactive_output(render_vdp_dashboard, {'mu_val': s_mu})
display(ui_vdp, out_vdp)

In [ ]:
# LAB 4.2: INTERACTIVE COBWEB MAPPER FOR 1D LOGISTIC MAP
display(HTML(create_dashboard_header(
    'LAB 4.2: THE COBWEB ANIMATION & DISCRETE STABILITY',
    'Track successive iterates: Fixed points (r < 3), 2-cycles (3 < r < 3.45), and chaotic wandering (r > 3.57).'
)))

def render_cobweb_dashboard(r_param, x0_init, n_steps):
    x_curve = np.linspace(0.0, 1.0, 400)
    y_curve = logistic_map(x_curve, r_param)
    x_web, y_web = compute_cobweb_points(r=r_param, x0=x0_init, n_iter=n_steps)
    
    fig, ax = plt.subplots(figsize=(8.5, 6.5), dpi=100)
    # Parabola & Diagonal
    ax.plot(x_curve, y_curve, color=PALETTE['cyan'], lw=2.0, label='f(x) = r*x*(1 - x)')
    ax.plot([0, 1], [0, 1], color=PALETTE['text_muted'], linestyle='--', lw=1.2, label='Identity Line y = x')
    # Cobweb trajectory
    ax.plot(x_web, y_web, color=PALETTE['crimson'], lw=1.3, alpha=0.85, label='Cobweb Iterates')
    ax.scatter([x0_init], [0], color=PALETTE['emerald'], s=70, zorder=6, label='Initial State x0')
    
    # Equilibrium evaluation
    x_star = 1.0 - (1.0 / r_param) if r_param > 1.0 else 0.0
    slope_star = r_param * (1.0 - 2.0 * x_star)
    stab_str = 'Stable (|f\'| < 1)' if abs(slope_star) < 1.0 else 'Unstable (|f\'| > 1)'
    
    ax.set_title('Cobweb Dynamics | r = ' + str(round(r_param, 3)) + ' | Fixed Point Slope: ' + str(round(slope_star, 2)) + ' (' + stab_str + ')', fontsize=11, pad=12)
    ax.set_xlabel('State x_n', fontsize=10)
    ax.set_ylabel('State x_{n+1}', fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc='upper left')
    plt.tight_layout()
    plt.show()

s_r = widgets.FloatSlider(value=2.8, min=1.0, max=4.0, step=0.02, description='Growth r:', layout=widgets.Layout(width='320px'))
s_x0 = widgets.FloatSlider(value=0.2, min=0.01, max=0.99, step=0.02, description='Initial x0:', layout=widgets.Layout(width='320px'))
s_n = widgets.IntSlider(value=40, min=5, max=120, step=5, description='Steps:', layout=widgets.Layout(width='250px'))

ui_cob = widgets.HBox([s_r, s_x0, s_n], layout=widgets.Layout(padding='10px', background_color=PALETTE['surface'], border='1px solid ' + PALETTE['border'], border_radius='8px'))
out_cob = widgets.interactive_output(render_cobweb_dashboard, {'r_param': s_r, 'x0_init': s_x0, 'n_steps': s_n})
display(ui_cob, out_cob)

In [ ]:
# LAB 4.3: FEIGENBAUM TREE & LYAPUNOV SPECTRUM SYNTHESIS
display(HTML(create_dashboard_header(
    'LAB 4.3: FEIGENBAUM CASCADE, PERIOD-3 & LYAPUNOV EXPONENT',
    'Direct alignment of bifurcation tree and Lyapunov exponent: Identify onset r_inf = 3.5699 and period-3 window.'
)))

r_tree, x_tree = generate_bifurcation_diagram(r_min=2.8, r_max=4.0, n_r=900, n_transient=300, n_sample=120)
r_lyap, lyap_spec = compute_lyapunov_spectrum(r_min=2.8, r_max=4.0, n_r=700, n_transient=250, n_iter=500)

fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True, dpi=100)

# Panel A: Bifurcation Tree
axs[0].scatter(r_tree, x_tree, s=0.2, color=PALETTE['cyan'], alpha=0.5)
axs[0].axvline(3.0, color=PALETTE['text_muted'], linestyle=':', lw=1.0)
axs[0].axvline(3.449, color=PALETTE['text_muted'], linestyle=':', lw=1.0)
axs[0].axvline(3.5699, color=PALETTE['crimson'], linestyle='--', lw=1.3, label='Accumulation Point r_inf = 3.5699')
axs[0].axvline(3.8284, color=PALETTE['amber'], linestyle='--', lw=1.3, label='Period-3 Window (r ~ 3.8284)')
axs[0].set_title('Feigenbaum Period-Doubling Bifurcation Tree (Universal delta = 4.6692, alpha = 2.5029)', fontsize=11)
axs[0].set_ylabel('Attractor State x*', fontsize=10)
axs[0].set_ylim(0, 1)
axs[0].legend(loc='upper left')

# Panel B: Lyapunov Exponent Spectrum
axs[1].plot(r_lyap, lyap_spec, color=PALETTE['emerald'], lw=1.2, label='Lyapunov Exponent lambda(r)')
axs[1].axhline(0, color=PALETTE['border'], linestyle='--', lw=1.2)
axs[1].axvline(3.5699, color=PALETTE['crimson'], linestyle='--', lw=1.3)
axs[1].axvline(3.8284, color=PALETTE['amber'], linestyle='--', lw=1.3)
axs[1].fill_between(r_lyap, lyap_spec, 0, where=(lyap_spec > 0), color=PALETTE['crimson'], alpha=0.25, label='Deterministic Chaos (lambda > 0)')
axs[1].fill_between(r_lyap, lyap_spec, 0, where=(lyap_spec <= 0), color=PALETTE['emerald'], alpha=0.2, label='Stable Periodic Cycles (lambda <= 0)')
axs[1].set_title('Discrete Lyapunov Exponent Spectrum | Quantitative Proof of Chaos', fontsize=11)
axs[1].set_xlabel('Control Parameter r', fontsize=10)
axs[1].set_ylabel('Lyapunov Exponent lambda', fontsize=10)
axs[1].set_xlim(2.8, 4.0)
axs[1].set_ylim(-1.5, 0.8)
axs[1].legend(loc='lower left')

plt.tight_layout()
plt.show()